# 03 — Camada Silver

## Objetivo

Nesta etapa vamos transformar os dados armazenados na camada **Bronze** em dados mais limpos, padronizados e confiáveis.

Nosso fluxo será:

`bronze.vendas → transformação → silver.vendas`

A camada Bronze continuará preservando os dados recebidos da fonte.

A camada Silver será responsável por aplicar regras de qualidade e padronização.

Nesta etapa vamos trabalhar com:

- leitura dos dados da Bronze;
- criação do schema Silver;
- remoção de duplicidades;
- tratamento de valores nulos;
- padronização de textos;
- validação de quantidades;
- validação de descontos;
- criação de campos derivados de data;
- validação do resultado final.

## Etapa 1 — Importação das bibliotecas

Antes de acessar o PostgreSQL, precisamos carregar as bibliotecas utilizadas pelo notebook.

Utilizaremos:

- `pandas`: manipulação e análise de dados;
- `psycopg2`: comunicação entre Python e PostgreSQL;
- `getpass`: entrada segura da senha do banco.

In [ ]:
# ==========================================================
# ETAPA 1 - IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

# Pandas será utilizado para manipulação
# e análise dos dados.
import pandas as pd


# Psycopg2 permite que o Python envie
# comandos SQL para o PostgreSQL.
import psycopg2


# getpass permite solicitar a senha do PostgreSQL
# sem deixar a senha escrita diretamente no notebook.
from getpass import getpass

## Etapa 2 — Configuração da conexão

Como estamos trabalhando em um novo notebook, precisamos novamente definir as informações necessárias para acessar o PostgreSQL.

Nosso banco está executando dentro do Docker.

A comunicação ocorre através da porta:

`localhost:5434`

O Docker encaminha essa conexão para a porta `5432` do PostgreSQL dentro do container.

In [ ]:
# ==========================================================
# ETAPA 2 - CONFIGURAÇÃO DA CONEXÃO
# ==========================================================

# Endereço do servidor PostgreSQL.
#
# Como o Docker está executando na nossa própria
# máquina, utilizamos localhost.
DB_HOST = "localhost"


# Porta externa configurada no docker-compose.yml.
#
# Computador: 5434
# Container:  5432
DB_PORT = 5434


# Nome do database utilizado pelo projeto.
DB_NAME = "db_vendas"


# Usuário utilizado para acessar o PostgreSQL.
DB_USER = "postgres"


# Solicita a senha durante a execução.
#
# getpass evita que a senha fique escrita
# diretamente no notebook.
DB_PASSWORD = getpass("Digite a senha do PostgreSQL: ")

## Etapa 3 — Teste de conexão

Antes de consultar ou transformar dados, vamos verificar se o Python consegue se comunicar corretamente com o PostgreSQL.

Esse teste evita continuar o pipeline quando existe algum problema de conexão.

In [ ]:
# ==========================================================
# ETAPA 3 - TESTE DE CONEXÃO
# ==========================================================

try:

    # Tenta estabelecer uma conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    print("✅ Conexão com PostgreSQL realizada com sucesso!")


except Exception as erro:

    print("❌ Não foi possível conectar ao PostgreSQL.")

    # Mostra apenas o tipo do erro.
    print("Tipo do erro:", type(erro).__name__)


finally:

    # Fecha a conexão após o teste.
    if "conexao" in locals():
        conexao.close()

## Etapa 4 — Leitura da Camada Bronze

A partir deste ponto, a fonte da nossa transformação será a tabela:

`bronze.vendas`

Primeiro vamos consultar alguns registros para confirmar que os dados estão disponíveis.

Utilizaremos:

- `SELECT`: seleciona dados;
- `FROM`: informa a tabela de origem;
- `LIMIT`: limita a quantidade de registros retornados.

In [ ]:
# ==========================================================
# ETAPA 4 - CONSULTA DA CAMADA BRONZE
# ==========================================================

try:

    # Abre a conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


    # Cria o cursor utilizado para executar SQL.
    cursor = conexao.cursor()


    # Consulta cinco registros da Bronze.
    cursor.execute("""
        SELECT *
        FROM bronze.vendas
        ORDER BY id_venda
        LIMIT 5;
    """)


    # fetchall() recupera todos os registros
    # retornados pela consulta.
    registros = cursor.fetchall()


    print("=== PRIMEIROS REGISTROS DA BRONZE ===\n")


    # Percorre os registros retornados.
    for registro in registros:
        print(registro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 5 — Regras de Transformação

Antes de criar a camada Silver, precisamos definir quais regras serão aplicadas aos dados.

### Regras utilizadas neste projeto

| Campo | Regra |
|---|---|
| id_venda | Não pode ser nulo |
| id_venda | Não deve possuir duplicidade |
| data_venda | Deve possuir uma data válida |
| cliente | Remover espaços extras |
| produto | Remover espaços extras |
| categoria | Padronizar texto |
| regiao | Padronizar texto |
| quantidade | Deve ser maior que zero |
| valor_unitario | Não pode ser negativo |
| desconto_percentual | Deve estar entre 0 e 100 |
| valor_total | Não pode ser negativo |
| forma_pagamento | Padronizar texto |

Também criaremos informações derivadas da data da venda:

- ano;
- mês;
- dia.

Esses campos facilitarão análises posteriores na camada Gold e no Apache Superset.

## Etapa 6 — Criação do Schema Silver

O PostgreSQL permite organizar tabelas utilizando schemas.

Já temos:

`bronze`

Agora criaremos:

`silver`

O schema Silver armazenará os dados após aplicação das regras de limpeza e padronização.

In [ ]:
# ==========================================================
# ETAPA 6 - CRIAÇÃO DO SCHEMA SILVER
# ==========================================================

try:

    # Abre a conexão.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


    # Cria o cursor.
    cursor = conexao.cursor()


    # Cria o schema Silver.
    #
    # IF NOT EXISTS permite executar novamente
    # a célula sem gerar erro caso ele já exista.
    cursor.execute("""
        CREATE SCHEMA IF NOT EXISTS silver;
    """)


    # Confirma a alteração no banco.
    conexao.commit()


    print("✅ Schema silver criado com sucesso!")


except Exception as erro:

    # Desfaz alterações caso algo dê errado.
    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar o schema Silver.")
    print("Tipo:", type(erro).__name__)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 7 — Criação da tabela Silver

Agora criaremos a tabela:

`silver.vendas`

Ela manterá as informações necessárias das vendas e adicionará campos derivados da data.

A estrutura Silver será diferente da Bronze porque agora estamos preparando os dados para utilização analítica.

In [ ]:
# ==========================================================
# ETAPA 7 - CRIAÇÃO DA TABELA SILVER.VENDAS
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # Cria a estrutura da tabela Silver.
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS silver.vendas (

            id_venda INTEGER,

            data_venda DATE,

            cliente TEXT,

            produto TEXT,

            categoria TEXT,

            regiao TEXT,

            quantidade INTEGER,

            valor_unitario NUMERIC(12,2),

            desconto_percentual INTEGER,

            valor_total NUMERIC(12,2),

            forma_pagamento TEXT,

            ano INTEGER,

            mes INTEGER,

            dia INTEGER
        );
    """)


    conexao.commit()

    print("✅ Tabela silver.vendas criada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar silver.vendas.")
    print("Tipo:", type(erro).__name__)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 8 — Transformação Bronze → Silver

Agora vamos realizar a transformação dos dados.

Os registros serão lidos de:

`bronze.vendas`

e inseridos em:

`silver.vendas`

Durante esse processo aplicaremos as regras definidas anteriormente.

A transformação será realizada utilizando SQL dentro do PostgreSQL.

In [ ]:
# ==========================================================
# ETAPA 8 - TRANSFORMAÇÃO BRONZE → SILVER
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 1. LIMPAR A SILVER
    # ------------------------------------------------------

    # Remove os registros existentes.
    #
    # Isso permite executar novamente o pipeline
    # sem duplicar os dados.
    cursor.execute("""
        TRUNCATE TABLE silver.vendas;
    """)


    # ------------------------------------------------------
    # 2. TRANSFORMAR E INSERIR
    # ------------------------------------------------------

    cursor.execute("""
        INSERT INTO silver.vendas (

            id_venda,
            data_venda,
            cliente,
            produto,
            categoria,
            regiao,
            quantidade,
            valor_unitario,
            desconto_percentual,
            valor_total,
            forma_pagamento,
            ano,
            mes,
            dia
        )

        SELECT DISTINCT

            id_venda,

            data_venda,

            TRIM(cliente),

            TRIM(produto),

            INITCAP(TRIM(categoria)),

            INITCAP(TRIM(regiao)),

            quantidade,

            valor_unitario,

            desconto_percentual,

            valor_total,

            INITCAP(TRIM(forma_pagamento)),

            EXTRACT(YEAR FROM data_venda)::INTEGER,

            EXTRACT(MONTH FROM data_venda)::INTEGER,

            EXTRACT(DAY FROM data_venda)::INTEGER

        FROM bronze.vendas

        WHERE
            id_venda IS NOT NULL

            AND data_venda IS NOT NULL

            AND quantidade > 0

            AND valor_unitario >= 0

            AND valor_total >= 0

            AND desconto_percentual BETWEEN 0 AND 100;
    """)


    # Confirma a transformação.
    conexao.commit()


    print("✅ Transformação Bronze → Silver concluída!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro durante a transformação.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 9 — Validação da Camada Silver

Após a transformação, precisamos verificar o resultado.

Vamos comparar a quantidade de registros existentes nas camadas Bronze e Silver.

Também verificaremos:

- IDs únicos;
- valores nulos;
- quantidades inválidas;
- descontos inválidos.

Essa etapa permite verificar se nossas regras de transformação foram aplicadas corretamente.

In [ ]:
# ==========================================================
# ETAPA 9 - VALIDAÇÃO DA CAMADA SILVER
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # QUANTIDADE NA BRONZE
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM bronze.vendas;
    """)

    total_bronze = cursor.fetchone()[0]


    # ------------------------------------------------------
    # QUANTIDADE NA SILVER
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas;
    """)

    total_silver = cursor.fetchone()[0]


    # ------------------------------------------------------
    # IDs ÚNICOS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(DISTINCT id_venda)
        FROM silver.vendas;
    """)

    ids_unicos = cursor.fetchone()[0]


    # ------------------------------------------------------
    # VALORES NULOS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas
        WHERE
            id_venda IS NULL
            OR data_venda IS NULL
            OR cliente IS NULL
            OR produto IS NULL
            OR valor_total IS NULL;
    """)

    valores_nulos = cursor.fetchone()[0]


    # ------------------------------------------------------
    # QUANTIDADES INVÁLIDAS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas
        WHERE quantidade <= 0;
    """)

    quantidades_invalidas = cursor.fetchone()[0]


    # ------------------------------------------------------
    # DESCONTOS INVÁLIDOS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas
        WHERE desconto_percentual NOT BETWEEN 0 AND 100;
    """)

    descontos_invalidos = cursor.fetchone()[0]


    # ------------------------------------------------------
    # RELATÓRIO
    # ------------------------------------------------------

    print("=== VALIDAÇÃO DA CAMADA SILVER ===\n")

    print("Registros Bronze:", total_bronze)
    print("Registros Silver:", total_silver)

    print("\nIDs únicos:", ids_unicos)

    print("\nProblemas encontrados:")
    print("Valores nulos:", valores_nulos)
    print("Quantidades inválidas:", quantidades_invalidas)
    print("Descontos inválidos:", descontos_invalidos)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()